# Paramétrage de l'environnement de travail et import des packages

In [66]:
import sys
from pathlib import Path

In [67]:
ROOT = Path.cwd().parents[0]

RAW_DATA = ROOT / "01_data" / "01_raw"
PROCESSED_DATA = ROOT / "01_data" / "02_processed"

%load_ext autoreload
%autoreload 2
sys.path.append(str(ROOT / "03_fonctions"))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [68]:
import pandas as pd
from fonctions_perso.geo_localisation import calcul_distances

# La base de données

**Import de la base de données**

In [69]:
data_fraud = (pd.read_parquet(RAW_DATA / "fraud_data.parquet")
.astype({
   "cc_num":"object",
   "zip":"object",
   "trans_date_trans_time":"datetime64[ns]",
   "dob":"datetime64[ns]"
})
.drop(["Unnamed: 0"], axis = 1))

**Renommage des variables**

In [70]:
data_fraud = data_fraud.rename(columns={
    "trans_date_trans_time":"date_heure_transaction",
    "cc_num":"numero_carte",
    "merchant":"nom_magasin",
    "category":"type_magasin",
    "amt":"montant_transaction",
    "first":"prenom",
    "last":"nom",
    "gender":"sexe",
    "street":"adresse_client",
    "city":"ville_client",
    "state":"etat_client",
    "zip":"code_postal_client",
    "lat":"latitude_domicile_client",
    "long":"longitude_domicile_client",
    "trans_num":"numero_transaction",
    "city_pop":"population_ville_client",
    "job":"profession_client",
    "dob":"date_naissance_client",
    "unix_time":"timestamp_unix_transacation",
    "merch_lat":"latitude_magasin",
    "merch_long":"longitude_magasin",
    "is_fraud":"target"
})

**Nettoyage de la variable "nom_magasin"**

In [71]:
data_fraud["nom_magasin"] = data_fraud["nom_magasin"].str.slice_replace(0,6,"")

**Informations globales**

In [72]:
data_fraud.info() # Pas de valeurs manquantes !

<class 'pandas.core.frame.DataFrame'>
Index: 563225 entries, 0 to 1295733
Data columns (total 22 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   date_heure_transaction       563225 non-null  datetime64[ns]
 1   numero_carte                 563225 non-null  object        
 2   nom_magasin                  563225 non-null  object        
 3   type_magasin                 563225 non-null  object        
 4   montant_transaction          563225 non-null  float64       
 5   prenom                       563225 non-null  object        
 6   nom                          563225 non-null  object        
 7   sexe                         563225 non-null  object        
 8   adresse_client               563225 non-null  object        
 9   ville_client                 563225 non-null  object        
 10  etat_client                  563225 non-null  object        
 11  code_postal_client           5

**Statistiques descriptives**

In [73]:
data_fraud[["montant_transaction","population_ville_client","target"]].describe()

,montant_transaction,population_ville_client,target
count,563225.000000,5.632250e+05,563225.000000
mean,75.548833,8.834256e+04,0.017135
std,170.529091,3.007564e+05,0.129775
min,1.000000,2.300000e+01,0.000000
25%,9.730000,7.410000e+02,0.000000
50%,47.840000,2.408000e+03,0.000000
75%,84.440000,1.968500e+04,0.000000
max,22768.110000,2.906700e+06,1.000000


# Feature engineering & Feature selection

**Création de nouvelles variables**

In [74]:
# Mois de la transaction
data_fraud["mois_transaction"] = data_fraud["date_heure_transaction"].dt.month_name()

# Jour de la transaction
data_fraud["jour_transaction"] = data_fraud["date_heure_transaction"].dt.day_name()

# Heure de la transaction
data_fraud["heure_transaction"] = data_fraud["date_heure_transaction"].dt.hour

# Distance "domicile" - "magasin"
calcul_distances(
    data_fraud,
    "latitude_domicile_client","longitude_domicile_client",
    "latitude_magasin","longitude_magasin",
    unit="km",
    out_col="distance_domicile_magasin"
)

# Age client
data_fraud["age_client"] = data_fraud["date_heure_transaction"].dt.year-data_fraud["date_naissance_client"].dt.year

**Suppression des variables non utilisables**

In [75]:
variables_a_supprimer = ["numero_carte","prenom","nom",
                         "sexe","adresse_client","code_postal_client",
                         "timestamp_unix_transacation","date_heure_transaction",
                         "date_naissance_client"]

data_fraud = data_fraud.drop(variables_a_supprimer, axis=1)

# Les variables suivantes ne seront pas utilisés ni pour l'EDA ni pour le modèle, elles sont donc supprimées

**Aperçu rapide des données**

In [77]:
data_fraud.head()

,nom_magasin,type_magasin,montant_transaction,ville_client,etat_client,latitude_domicile_client,longitude_domicile_client,population_ville_client,profession_client,numero_transaction,latitude_magasin,longitude_magasin,target,mois_transaction,jour_transaction,heure_transaction,distance_domicile_magasin,age_client
0,Kirlin and Sons,personal_care,2.86,Columbia,SC,33.9659,-80.9355,333497,Mechanical engineer,2da90c7d74bd46a0caf3777415b3ebd3,33.986391,-81.200714,0,June,Sunday,12,24.561496,52
1,Sporer-Keebler,personal_care,29.84,Altonah,UT,40.3207,-110.4360,302,"Sales professional, IT",324cc204407e99f51b0d6ca0055005e7,39.450498,-109.960431,0,June,Sunday,12,104.925237,30
2,"Swaniawski, Nitzsche and Welch",health_fitness,41.28,Bellmore,NY,40.6729,-73.5365,34496,"Librarian, public",c81755dbbbea9d5c77f094348a7579be,40.495810,-74.196111,0,June,Sunday,12,59.080159,50
3,Haley Group,misc_pos,60.05,Titusville,FL,28.5697,-80.8191,54767,Set designer,2159175b9efe66dc301f149d3d5abf8c,28.812398,-80.883061,0,June,Sunday,12,27.698606,33
4,Johnston-Casper,travel,3.19,Falmouth,MI,44.2529,-85.0170,1126,Furniture designer,57ff021bd3f328f8738bb535c302a31b,44.959148,-85.884734,0,June,Sunday,12,104.335250,65


**Export des données**

In [79]:
data_fraud.to_parquet(PROCESSED_DATA / "data_fraud_processed.parquet", index = False)